# Simulation viewer for PyPSA-Earth Taiwan

This notebook is a first-pass results dashboard for the completed Taiwan run.

PyPSA-Earth's local checkout does not include a result-analysis notebook, but the tutorial docs point to the external documentation example `sample_network_analysis.ipynb` in `pypsa-meets-earth/documentation`.

It loads the solved network from:

`../results/networks/elec_s_6_ec_lcopt_Co2L-4H.nc`

## What To Show

- Run summary: objective value, snapshots, buses, lines, generators, stores, and storage units.
- Capacity mix: installed/optimized generator capacity by carrier.
- Energy mix: total generated electricity by carrier as an interactive Plotly pie chart.
- Demand and dispatch: interactive time series showing total load and carrier-level dispatch.
- Renewable curtailment: estimated available renewable energy minus dispatched renewable energy.
- Spatial overview: interactive Plotly map with OpenStreetMap basemap, Taiwan buses, lines, and generator capacity bubbles.
- Sanity checks: load served, missing values, suspicious negative dispatch, and solved output path.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents]
REPO = next((candidate for candidate in repo_candidates if (candidate / "Snakefile").exists()), None)
if REPO is None:
    raise FileNotFoundError("Could not find repository root containing Snakefile.")

VIEWER_DIR = REPO / "pypsa_tw" / "viewer"
sys.path.insert(0, str(VIEWER_DIR))

try:
    from viewer_helper import (
        build_capacity_comparison,
        build_curtailment_summary,
        build_dispatch_plot_data,
        build_generation_comparison,
        build_bus_technology_limits,
        build_model_inputs_summary,
        build_run_summary,
        build_sanity_checks_by_run,
        list_available_results,
        load_selected_runs,
        plot_capacity_comparison,
        plot_dispatch,
        plot_generation_comparison,
        plot_spatial_overview,
        plot_transmission_network,
        build_transmission_assets,
        build_transmission_summary,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook requires `plotly`, `pypsa`, and the local `viewer_helper.py` module. "
        "Install missing packages in the pypsa-earth environment before running it."
    ) from exc

pd.options.display.float_format = "{:,.2f}".format

RESULTS_ROOT = REPO / "results"
RESULTS_ROOT


WindowsPath('../results')

In [2]:
## show all the available reulst and a list
available_results, available_results_df = list_available_results(RESULTS_ROOT)

if available_results_df.empty:
    raise FileNotFoundError(
        f"No result folders with network files were found under {RESULTS_ROOT.resolve()}"
    )

available_results_df

,result_index,result_folder,network_file
0,0,tw_basic_test1_2013_7d_4h_6b,..\results\tw_basic_test1_2013_7d_4h_6b\networ...
1,1,tw_test1_highs_2013_7d_4h_6b,..\results\tw_test1_highs_2013_7d_4h_6b\networ...
2,2,tw_test2_highs_2013_fullyear_4h_6b,..\results\tw_test2_highs_2013_fullyear_4h_6b\...


In [3]:
## select a result folder to load the network from the above list
SELECTED_RESULT_INDICES = [0, 1]

selected_results, run_result_paths, run_networks = load_selected_runs(
    available_results, SELECTED_RESULT_INDICES
)

selected_result = selected_results[1]
RESULT_PATH = run_result_paths[selected_result.name]
n = run_networks[selected_result.name]

pd.DataFrame(
    {
        "result_folder": list(run_result_paths.keys()),
        "result_file": [str(path) for path in run_result_paths.values()],
        "used_for_single_run_sections": [name == selected_result.name for name in run_result_paths.keys()],
    }
)

INFO:pypsa.io:Imported network elec_s_6_ec_lcopt_Co2L-4H.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
INFO:pypsa.io:Imported network elec_s_6_ec_lcopt_Co2L-4H.nc has buses, carriers, generators, global_constraints, lines, loads, storage_units


,result_folder,result_file,used_for_single_run_sections
0,tw_basic_test1_2013_7d_4h_6b,..\results\tw_basic_test1_2013_7d_4h_6b\networ...,False
1,tw_test1_highs_2013_7d_4h_6b,..\results\tw_test1_highs_2013_7d_4h_6b\networ...,True


## Run Summary

This is the quick health check: did we load the solved file, how large is it, and what objective value did the optimization report?

In [4]:
summary_df = build_run_summary(run_networks, run_result_paths)
summary_df

,run,result_file,objective,snapshots,first_snapshot,last_snapshot,buses,lines,generators,loads,stores,storage_units
0,tw_basic_test1_2013_7d_4h_6b,..\results\tw_basic_test1_2013_7d_4h_6b\networ...,"176,236,517,574.60",42,2013-03-01,2013-03-07 20:00:00,18,4,52,6,12,4
1,tw_test1_highs_2013_7d_4h_6b,..\results\tw_test1_highs_2013_7d_4h_6b\networ...,"685,540,518,198.47",42,2013-03-01,2013-03-07 20:00:00,6,4,40,6,0,4


## Model Inputs

This section summarizes model-side inputs exposed in the solved networks, starting with global constraints such as CO2 caps and other system-wide limits.

### Global Constraints

In [5]:
model_inputs_df = build_model_inputs_summary(run_networks)
model_inputs_df

,run,constraint_name,type,carrier_attribute,sense,constant,mu
0,tw_basic_test1_2013_7d_4h_6b,CO2Limit,primary_energy,co2_emissions,<=,"41,567.19","-13,476.85"
1,tw_test1_highs_2013_7d_4h_6b,CO2Limit,primary_energy,co2_emissions,<=,inf,NaN


### Bus Technology Limits

This table shows possible technologies by bus in the selected runs, together with their aggregated minimum and maximum capacity bounds.

In [6]:
bus_technology_limits_df = build_bus_technology_limits(run_networks)
bus_technology_limits_df

,run,component,bus,carrier,extendable,capacity_min_MW,capacity_max_MW
0,tw_basic_test1_2013_7d_4h_6b,Generator,TW0 0,CCGT,False,"2,278.00","2,278.00"
1,tw_basic_test1_2013_7d_4h_6b,Generator,TW0 0,coal,False,"5,270.00","5,270.00"
2,tw_basic_test1_2013_7d_4h_6b,Generator,TW0 0,load shedding,False,0.00,inf
3,tw_basic_test1_2013_7d_4h_6b,Generator,TW0 0,offwind-ac,True,"1,712.00","4,632.87"
4,tw_basic_test1_2013_7d_4h_6b,Generator,TW0 0,offwind-dc,True,0.00,"12,360.93"
...,...,...,...,...,...,...,...
119,tw_test1_highs_2013_7d_4h_6b,Generator,TW2 0,solar,False,796.10,"1,847.66"
120,tw_test1_highs_2013_7d_4h_6b,StorageUnit,TW0 1,hydro,False,0.00,inf
121,tw_test1_highs_2013_7d_4h_6b,StorageUnit,TW0 2,PHS,False,0.00,inf
122,tw_test1_highs_2013_7d_4h_6b,StorageUnit,TW0 2,hydro,False,0.00,inf


### Transmission Network

This summarizes the AC transmission lines in each selected run, including connected buses, length, voltage, capacity, and whether each line is extendable.

In [13]:
transmission_assets_df = build_transmission_assets(run_networks)
transmission_summary_df = build_transmission_summary(transmission_assets_df)

display(transmission_summary_df)
display(transmission_assets_df)

NameError: name 'build_transmission_assets' is not defined

### Transmission Map

This map shows buses and AC transmission lines for the selected single-run view. Line labels show capacity in GW.

In [ ]:
plot_transmission_network(n, selected_result.name).show()

## Model Outputs

### Capacity Mix

This shows optimized installed generator capacity by carrier. When optimized capacity is unavailable for fixed generators, the notebook falls back to nominal capacity.

In [7]:
capacity_compare_df, capacity_df = build_capacity_comparison(run_networks)
display(capacity_compare_df)

plot_capacity_comparison(capacity_df).show()


carrier,load shedding,solar,onwind,CCGT,coal,offwind-dc,offwind-ac,nuclear,ror,oil
run,,,,,,,,,,
tw_basic_test1_2013_7d_4h_6b,48.72,40.43,42.89,18.32,18.24,15.93,11.71,5.32,1.00,0.28
tw_test1_highs_2013_7d_4h_6b,48.72,12.42,1.10,18.32,18.24,0.00,2.22,5.32,1.00,0.28


### Energy Mix

Energy is weighted by the model snapshot weighting, so the totals represent the modeled period rather than just raw summed time steps. The pie chart shows each carrier's share of modeled electricity generation.

In [8]:
snapshot_weight = n.snapshot_weightings.generators.reindex(n.snapshots).fillna(1.0)

generation_by_run, generation_compare_df, energy_mix_df = build_generation_comparison(run_networks)
display(generation_compare_df)

plot_generation_comparison(energy_mix_df).show()

generation_gwh = generation_by_run[selected_result.name].sort_values(ascending=False)


carrier,onwind,coal,solar,offwind-dc,nuclear,offwind-ac,CCGT,load shedding,ror,oil
run,,,,,,,,,,
tw_basic_test1_2013_7d_4h_6b,"152,954.48",0.00,"89,166.18","89,255.84","36,734.57","55,944.23",121.76,0.00,"1,484.96",0.00
tw_test1_highs_2013_7d_4h_6b,"4,030.25","155,351.24","26,997.85",0.00,"42,857.00","11,508.73","55,900.94","6,776.44","1,484.96",0.00


### Demand And Dispatch

This compares total load with aggregated generator dispatch by carrier for the selected single-run view. It is useful for spotting whether the solved network is serving demand and which carriers dominate each period.

In [9]:
load, dispatch_long = build_dispatch_plot_data(n)
plot_dispatch(load, dispatch_long, selected_result.name).show()

### Renewable Curtailment

This estimates available renewable output from `p_max_pu * p_nom_opt` and compares it with dispatched output for the selected single-run view. It is a useful early signal of overbuild, congestion, or insufficient flexibility.

In [10]:
curtailment_summary = build_curtailment_summary(n, snapshot_weight)

display(
    curtailment_summary.rename(columns={
        "available_MWh": "available_GWh",
        "dispatched_MWh": "dispatched_GWh",
        "curtailed_MWh": "curtailed_GWh",
    }).assign(run=selected_result.name).reset_index()
)

,carrier,available_GWh,dispatched_GWh,curtailed_GWh,curtailment_rate_pct,run
0,offwind-ac,"11,508.73","11,508.73",0.00,0.00,tw_test1_highs_2013_7d_4h_6b
1,offwind-dc,0.00,0.00,0.00,<NA>,tw_test1_highs_2013_7d_4h_6b
2,onwind,"4,102.37","4,030.25",72.12,1.76,tw_test1_highs_2013_7d_4h_6b
3,ror,"1,484.96","1,484.96",0.00,0.00,tw_test1_highs_2013_7d_4h_6b
4,solar,"27,904.18","26,997.85",906.33,3.25,tw_test1_highs_2013_7d_4h_6b


### Spatial Overview

A quick geographic check for the selected single-run view: buses and transmission lines should roughly cover Taiwan, and capacity bubbles should sit on modeled buses.

In [11]:
plot_spatial_overview(n, selected_result.name).show()

f:\Barton\Repositories\pypsa-earth\pypsa_tw\viewer_helper.py:295: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\_plotly_utils\basevalidators.py:2669: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  trace = self.get_trace_class(trace_type)(
f:\Barton\Repositories\pypsa-earth\pypsa_tw\viewer_helper.py:295: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(
f:\Barton\Repositories\pypsa-earth\.venv\Lib\site-packages\_plotly_utils\basevalidators.py:2669: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  trace = self.get_trace_class(trace_t

### Sanity Checks

These checks are not a full validation report, but they catch common early issues across all selected runs: missing solved outputs, NaNs, unserved demand hints, and unusual negative generation.

In [12]:
build_sanity_checks_by_run(run_networks, run_result_paths, generation_by_run)

,run,result_file_exists,objective_is_finite,load_total_GWh,generation_total_GWh,generator_dispatch_has_nan,negative_generator_dispatch_MWh
0,tw_basic_test1_2013_7d_4h_6b,True,True,"307,443.44","425,662.03",False,0.00
1,tw_test1_highs_2013_7d_4h_6b,True,True,"307,443.44","304,907.42",False,0.00
